<a href="https://colab.research.google.com/github/danof2morrow/data-science-portfolio/blob/main/DSC670_week5_Morrow_D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Chat Using a Wikipedia Article

## Introduction

This notebook builds a retrieval-augmented generation (RAG) chat application using a Wikipedia article as the external knowledge source. The structure follows the RAG example from this week's reading, but instead of extracting text from a PDF, the text will come from a Wikipedia page.

RAG combines information retrieval with a language model. When a question is asked, the system first searches the external text for information related to the question. That information is then provided to the language model as context for generating its response. This allows the model to base its answer on an outside source rather than relying only on information learned during training.

For this assignment, I will be using the Wikipedia page for the best cinematic trilogy ever created, *The Lord of the Rings*. The article covers the story, characters, publication history, themes, reception, and adaptations, which should give the retrieval process a ton of information to search when different questions are asked.

The goal is to retrieve the most relevant information from the article and use it to help the language model answer questions about *The Lord of the Rings*.

In [2]:
!pip install -q beautifulsoup4 sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 100.2 MB/s eta 0:00:00


## Retrieve the Wikipedia Article

The original RAG example from the reading used a PDF as the external source. For this, I’ll be replacing the PDF with the Wikipedia article for the most significant and amazing movie of all time and space, *The Lord of the Rings*. The source format is different, but the outome should be the same: convert the source into clean text that can later be divided into chunks and searched.

I’ll be using Beautiful Soup to extract the article text from the webpage.

In [3]:
import requests
from bs4 import BeautifulSoup
import re

#Wikipedia article
url = "https://en.wikipedia.org/wiki/The_Lord_of_the_Rings_(film_series)"

#Request the webpage
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
response.raise_for_status()

#Parse the webpage
soup = BeautifulSoup(response.text, "html.parser")

#Extract paragraph text from the main Wikipedia content
paragraphs = soup.select(".mw-parser-output p")

#Fallback if Wikipedia's page structure is different
if not paragraphs:
    paragraphs = soup.find_all("p")

article_text = " ".join(
    paragraph.get_text(" ", strip=True)
    for paragraph in paragraphs
    if paragraph.get_text(strip=True)
)

#Remove citation markers and extra spaces
article_text = re.sub(r"\[[^\]]+\]", "", article_text)
article_text = re.sub(r"\s+", " ", article_text).strip()

#Output
print(f"Characters extracted: {len(article_text):,}")
print(f"Words extracted: {len(article_text.split()):,}")
print("\nSample text:\n")
print(article_text[:1500])

Characters extracted: 39,958
Words extracted: 6,783

Sample text:

The Lord of the Rings is a trilogy of epic fantasy films directed by Peter Jackson . The films are based on the novel by J. R. R. Tolkien , drawing their titles and most of their plots from its three volumes : The Fellowship of the Ring (2001), The Two Towers (2002), and The Return of the King (2003). Produced and distributed by New Line Cinema with the co-production of Jackson's WingNut Films , the films feature an ensemble cast . Set in the fictional world of Middle-earth , the films follow the hobbit Frodo Baggins as he and the Company of the Ring set out to destroy the One Ring , an artifact of great magical power, and defeat its maker, the Dark Lord Sauron . The Company eventually separates, and Frodo continues the quest with his loyal valet Sam and the treacherous Gollum . Meanwhile, Aragorn , heir in exile to the throne of Gondor , along with the elf Legolas , the dwarf Gimli , other hobbits Merry and Pippin , an

### Output

The extraction returned a large amount of text, and it did confirm that the correct Wikipedia page was captured. It includes information about the films, production, cast, filming, budget, and release history, so there should be enough variety to test whether the retrieval process can distinguish between different types of questions.

One limitation I'm seeing is that extracting only the paragraph text removes some of the original structure of the Wikipedia page, including the section headings. That means the retriever will have to rely mostly on the meaning of the text itself rather than clues from the article's organization. I think it's a reasonable tradeoff since it keeps the input simple and close to the text-based RAG example from the reading, but we will see.

## Chunking Strategy

I’ll start with 500-character chunks and a 100-character overlap to stay close to the example. The overlap should help preserve information that falls near the boundary between two chunks, even though it introduces some repeated text. I’ll then look at both the number of chunks and an example chunk to see whether the size looks large enough to preserve meaning without becoming so large that everything gets lost in translation.

In [4]:
#Split the article into overlapping text chunks
def chunk_text(text, max_chars=500, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + max_chars
        chunk = text[start:end]
        chunks.append(chunk)
        start += max_chars - overlap

    return chunks

chunks = chunk_text(article_text)

#Output
print(f"Total chunks created: {len(chunks)}")
print("\nFirst chunk:\n")
print(chunks[0])

Total chunks created: 100

First chunk:

The Lord of the Rings is a trilogy of epic fantasy films directed by Peter Jackson . The films are based on the novel by J. R. R. Tolkien , drawing their titles and most of their plots from its three volumes : The Fellowship of the Ring (2001), The Two Towers (2002), and The Return of the King (2003). Produced and distributed by New Line Cinema with the co-production of Jackson's WingNut Films , the films feature an ensemble cast . Set in the fictional world of Middle-earth , the films follow th


### Output

The article divided into 100 chunks, which gives the retrieval system much smaller sections to search instead of treating the entire Wikipedia page as one document.

The first chunk did enforce a limitation of fixed-length chunking. It ends in the middle of a sentence, not ideal since it means the 500-character boundary doesn’t always match the structure of the text but the 100-character overlap should help recover some of that lost context in the next chunk.

 I’ll keep the fixed-length approach because it matches the example from the reading and gives a clear baseline for evaluatoon. If retrieval quality becomes a problem later, I may try another approach.

## Create Embeddings

RAG depends on being able to find text that is related to a question even when the wording isn’t exactly the same. To make that possible, each text chunk is converted into an embedding, which represents the semantic meaning of the text as a numerical vector. Chunks with similar meanings should end up closer together in this vector space, allowing the retrieval system to search by meaning rather than relying only on exact keywords.

I’ll use the same Sentence Transformer model from the assignment, `all-MiniLM-L6-v2`. Keeping the embedding model consistent gives me a useful baseline because I’m changing the source from a PDF to a Wikipedia article without also changing the retrieval model.

At this stage, creating the embeddings doesn’t tell me whether the system will retrieve the correct information. It only creates the numerical representations that make semantic comparison possible. I’ll evaluate their usefulness later by looking at which chunks are actually returned for different questions.

In [5]:
from sentence_transformers import SentenceTransformer

#Load the same embedding model used in the RAG example
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

#Create an embedding for each text chunk
embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True
)

#Output
print(f"Embedding matrix shape: {embeddings.shape}")
print(f"Embedding dimensions per chunk: {embeddings.shape[1]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding matrix shape: (100, 384)
Embedding dimensions per chunk: 384


### Output

The 100 text chunks produced 100 embeddings, so every chunk now has its own numerical representation. Each embedding contains 384 dimensions, which is determined by the `all-MiniLM-L6-v2` model. These dimensions shouldn’t be interpreted as individual features like words or topics. Together, they represent patterns the model has learned about the meaning of the text.

The result confirms that the entire article was successfully converted into a consistent vector format that can be searched. However, creating the embeddings doesn’t tell me whether they represent the chunks well enough for this RAG system. Their value depends on whether questions with similar meaning are placed close enough to the correct chunks for FAISS to retrieve them.


## Build the FAISS Vector Index

The embeddings give each chunk a numerical representation, but I still need a way to compare those vectors with a user's question. I'll use FAISS, which is the same vector-search approach from the assignment.

Because the embeddings were normalized, FAISS can rank the chunks by similarity to the question embedding. This should allow the system to search by meaning instead of relying only on exact keyword matches.

For now, I’m mainly confirming that all 100 chunk embeddings are stored in the index. The real test will be whether the most relevant chunks are actually retrieved for different questions.

In [6]:
import faiss

#Create the FAISS index
index = faiss.IndexFlatIP(embeddings.shape[1])

#Add the article embeddings
index.add(embeddings)

#Output
print(f"Chunks available: {len(chunks)}")
print(f"Vectors stored in FAISS: {index.ntotal}")

Chunks available: 100
Vectors stored in FAISS: 100


### Output

FAISS stored all 100 chunk embeddings, so the index contains the full set of text chunks from the article. This confirms that the vector index was created successfully and is ready to be used for similarity searches.

The index is complete, now let's hope it retrieves the right chunks when questions are asked.

## Test the Retrieval

Before adding the language model, I want to test the retrieval step by itself. This should show if the vector search is actually finding relevant parts of the Wikipedia article for a question.

I’ll start with a question that should have a clear answer in the article. If the returned chunks are relevant, that gives me more confidence that the embeddings and FAISS index are working as intended. If they aren’t, the issue could be with retrieval rather than the language model.

In [7]:
#Retrieve the most relevant chunks for a question
def retrieve_chunks(question, top_k=3):
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(question_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": score,
            "chunk": chunks[idx]
        })

    return results

#Test retrieval
test_question = "Where were The Lord of the Rings movies filmed?"
retrieved = retrieve_chunks(test_question)

for i, result in enumerate(retrieved, 1):
    print(f"\nResult {i} - Similarity score: {result['score']:.3f}")
    print(result["chunk"])


Result 1 - Similarity score: 0.632
exposure in the series, with the country's tourism industry waking up to an audience's familiarity. In 2002, the Museum of New Zealand Te Papa Tongarewa in Wellington created a travelling exhibition of the film trilogy, The Lord of the Rings Motion Picture Trilogy: The Exhibition . It visited cities around the world including Boston, London, Sydney, Singapore, and Houston. The Lord of the Rings left a legacy of court cases over profits from the trilogy. Sixteen cast members (Noel Appleby, Jed Br

Result 2 - Similarity score: 0.557
The Lord of the Rings is a trilogy of epic fantasy films directed by Peter Jackson . The films are based on the novel by J. R. R. Tolkien , drawing their titles and most of their plots from its three volumes : The Fellowship of the Ring (2001), The Two Towers (2002), and The Return of the King (2003). Produced and distributed by New Line Cinema with the co-production of Jackson's WingNut Films , the films feature an ensembl

### Output

The retrieval results are interesting. The first result mentions New Zealand, but the surrounding text is mainly about tourism and an exhibition rather than where the films were actually shot. The other two results have even to do with the question.

Even a successful vector search doesn’t automatically mean the most useful information is going to be retrieved. I'm wondering if the fixed-length chunks are part of the problem because information about filming locations can be separated from the surrounding context. The wording of the question may also affect which chunks are considered most similar.

Before using these results as context for the language model, I'm going to test whether a more specific question can improve the results.

In [8]:
test_question = "In what country were all three Lord of the Rings films shot?"

retrieved = retrieve_chunks(test_question)

for i, result in enumerate(retrieved, 1):
    print(f"\nResult {i} - Similarity score: {result['score']:.3f}")
    print(result["chunk"])


Result 1 - Similarity score: 0.656
exposure in the series, with the country's tourism industry waking up to an audience's familiarity. In 2002, the Museum of New Zealand Te Papa Tongarewa in Wellington created a travelling exhibition of the film trilogy, The Lord of the Rings Motion Picture Trilogy: The Exhibition . It visited cities around the world including Boston, London, Sydney, Singapore, and Houston. The Lord of the Rings left a legacy of court cases over profits from the trilogy. Sixteen cast members (Noel Appleby, Jed Br

Result 2 - Similarity score: 0.573
The Lord of the Rings is a trilogy of epic fantasy films directed by Peter Jackson . The films are based on the novel by J. R. R. Tolkien , drawing their titles and most of their plots from its three volumes : The Fellowship of the Ring (2001), The Two Towers (2002), and The Return of the King (2003). Produced and distributed by New Line Cinema with the co-production of Jackson's WingNut Films , the films feature an ensembl

### Output

Changing the wording of the question did improve similarity scores, but the retrieval still didn’t return the strongest passage from the article. And the third result does directly connect the films with New Zealand, but the first result is mostly about tourism and the second is general background about the trilogy.

Similarity scores aren’t the same thing as quality answers. A chunk can be mathematically similar to the question without containing the clearest evidence needed to answer it. The fixed-length chunking may also be affecting the results because related sentences can be separated across chunk boundaries.

Even with that limitation, the retrieved context contains enough information to identify New Zealand, so I’ll continue with the RAG pipeline and see how the language model handles the retrieved information.

## Adding the Language Model

I’ll be using the same FLAN-T5 model used in the assigned RAG example. The current version of the Hugging Face Transformers library no longer supports the `text2text-generation` pipeline used in the reading, so I’ll load the model directly instead.

The model itself hasn’t changed, so the purpose is still the same: combine the retrieved Wikipedia context with a question and hopefully generate an answer grounded in that information.

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

#Use the same FLAN-T5 model from the assigned RAG example
model_name = "google/flan-t5-large"

#Use the GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"

#Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
qa_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

#Move model to the available device
qa_model = qa_model.to(device)

#Output
print(f"FLAN-T5 model loaded successfully on {device}.")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 model loaded successfully on cuda.


### Output

FLAN-T5 loaded successfully on the GPU, so the generation portion of the RAG system is ready to use. Now the model can receive retrieved Wikipedia context and generate answers from it.

## Combine Retrieval and Generation

The next step is to connect the two parts of the RAG system. When a question is asked, FAISS will first retrieve the three chunks that are most similar to the question. Those chunks will then become the context given to FLAN-T5.

I’ll tell the model to answer only from the retrieved context and to say when the information isn’t available. This is important because RAG is intended to ground the response in the external source, but the quality of the final answer still depends on whether the retrieval step finds useful information.

In [10]:
#Combine retrieval and generation
def answer_question(question, top_k=3):
    retrieved = retrieve_chunks(question, top_k)

    context = "\n\n".join(
        result["chunk"] for result in retrieved
    )

    prompt = f"""
Answer the question using only the context below.
If the answer is not available in the context, say that the information is not available.

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = qa_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer, retrieved


#Test the complete RAG pipeline
test_question = "Who directed The Lord of the Rings film trilogy?"

answer, retrieved = answer_question(test_question)

#Output
print(f"Question: {test_question}")
print(f"Answer: {answer}")

Question: Who directed The Lord of the Rings film trilogy?
Answer: Peter Jackson


### Output

The RAG test worked as expected. The system returned Peter Jackson, which is directly supported by the Wikipedia article.

This is great because it confirms that the full pipeline is functioning from end to end: the question is converted into an embedding, FAISS retrieves related chunks, and FLAN-T5 uses those chunks to generate a response. However, this was a pretty easy questions since the director is mentioned near the beginning of the article.

The next test won't be as easy and the questions will depend on less obvious parts of the article to see whether retrieval remains accurate.

In [11]:
test_question = "How long did it take to make The Lord of the Rings?"

answer, retrieved = answer_question(test_question)

#Output
print(f"Question: {test_question}")
print(f"Answer: {answer}")

Question: How long did it take to make The Lord of the Rings?
Answer: the information is not available


### Output

When I asked, "How long did it take to make The Lord of the Rings?", the system said the information wasn’t available. Which is unfortunate but if we look at the retrieved chunks it helps explains why. The top results focused on editing, release history, and the film score rather than the actual production timeline.

This tell me that not getting the right results occurred during retrieval rather than generation. FLAN-T5 was instructed to answer only from the retrieved context, and because that particular context didn’t contain the answer, it (correctly) avoided making one up.

The question seems fairly natural for a person to ask, but the phrase "make The Lord of the Rings" is broad enough that the embedding search connected it with several parts of the filmmaking process. I'll try a more specific wording and see if it improves the results.

In [12]:
test_question = "How long did filming take for The Lord of the Rings?"

answer, retrieved = answer_question(test_question)

#Output
print(f"Question: {test_question}")
print(f"Answer: {answer}")

for i, result in enumerate(retrieved, 1):
    print(f"\nResult {i} - Similarity score: {result['score']:.3f}")
    print(result["chunk"])

Question: How long did filming take for The Lord of the Rings?
Answer: the information is not available

Result 1 - Similarity score: 0.644
re edited in consecutive years, although Selkirk continued to act as "Supervising Editor" on the first two entries. Daily rushes would often last up to four hours, and by the time The Fellowship of the Ring had been released, assembly cuts of the other two films (4 1 ⁄ 2 hours each) were already prepared. In total, 1,828 km (six million feet) of film was edited down to the 11 hours and 26 minutes (686 minutes) of extended running time. Howard Shore composed, orchestrated, conducted and produced 

Result 2 - Similarity score: 0.583
. The extended cuts of the films and the supplemental special features (dubbed "appendices") were spread over two discs each in a four-disc box set for each film. A limited collector's edition was also released featuring sculpted bookends. The Fellowship of the Ring was released on 12 November 2002, containing 30 minutes 

### Output

The more specific question still didn't produce the correct answer. The retrieved chunks focused on editing and extended editions rather than the production timeline, so FLAN-T5 again responded that the information was not available.

I checked the source text to confirm that the answer was actually present and it is. The article states that principal photography took place from October 1999 through December 2000, so the problem isn't missing source information. The correct chunk ranked 26th in the similarity search with a score of 0.428, which means it never reached the language model when only the top three results were used.

This shows that there's a limitation in the retrieval step. A question can be relatively simple and the answer can exist in the source material, but the embedding search may still fail to connect the wording of the question with the wording used in the article. This is definetly an art.

In [13]:
test_question = "When did principal photography take place for The Lord of the Rings?"

answer, retrieved = answer_question(test_question)

#Output
print(f"Question: {test_question}")
print(f"Answer: {answer}")

for i, result in enumerate(retrieved, 1):
    print(f"\nResult {i} - Similarity score: {result['score']:.3f}")
    print(result["chunk"])

Question: When did principal photography take place for The Lord of the Rings?
Answer: 2001

Result 1 - Similarity score: 0.579
exposure in the series, with the country's tourism industry waking up to an audience's familiarity. In 2002, the Museum of New Zealand Te Papa Tongarewa in Wellington created a travelling exhibition of the film trilogy, The Lord of the Rings Motion Picture Trilogy: The Exhibition . It visited cities around the world including Boston, London, Sydney, Singapore, and Houston. The Lord of the Rings left a legacy of court cases over profits from the trilogy. Sixteen cast members (Noel Appleby, Jed Br

Result 2 - Similarity score: 0.516
King its Best Picture Award at the 2003 Awards Ceremony , hosted by Andrew Johnston , chair of the organisation at that time, who called it "a masterful piece of filmmaking." Commentators have compared Jackson's film trilogy with Tolkien's written work, remarking that while both have been extremely successful commercially, they diffe

### Output

Using the phrase "principal photography" didn't retrieve the section that actually contains the filming dates. The top three results were about tourism, critical reception, and general information about the trilogy rather than the production timeline.

The generated answer was also incorrect. FLAN-T5 responded with "2001," even though none of the retrieved passages identified that as the date of principal photography. One of the chunks did contain 2001 as the release year of *The Fellowship of the Ring*, so the model appears to have used an unrelated date from the available context.

This shows that RAG can fail in more than one place. The retriever may fail to supply the correct information, and the language model may still attempt to produce an answer from weak context rather than admitting that the answer is unavailable. The earlier grounding instruction reduced this behavior in some cases, but it didn't eliminate it.

## Create the Interactive RAG Chat

The individual tests show that the retrieval and generation steps can work together, although they’ve also revealed how it can be a little wonky in retrieval quality. The final step is to place the same process into an interactive loop so I can ask different questions without running a new code cell each time.

Each question will still go through the same RAG process independently. The question is used to retrieve relevant Wikipedia chunks, and FLAN-T5 generates an answer from that context. This is a simple chat interface rather than a conversational system with memory, which keeps it consistent with the structure used in the assignment.

In [15]:
#Interactive RAG chat
while True:
    question = input("Ask a question about The Lord of the Rings (or type 'exit'): ")

    if question.lower().strip() == "exit":
        print("Chat ended.")
        break

    answer, retrieved = answer_question(question)

    #Output
    print(f"\nAnswer: {answer}\n")

Ask a question about The Lord of the Rings (or type 'exit'): Who composed the music for LOTR?

Answer: Howard Shore

Ask a question about The Lord of the Rings (or type 'exit'): What is Bilbo Baggins?

Answer: hobbit

Ask a question about The Lord of the Rings (or type 'exit'): How many awards did Lord of the Rings win?

Answer: 17

Ask a question about The Lord of the Rings (or type 'exit'): what was peter jacksons favorite food while filming lord of the rings?

Answer: the information is not available

Ask a question about The Lord of the Rings (or type 'exit'): exit
Chat ended.


### Output

The interactive chat handled several different question styles fairly well. When I asked who composed the music for "LOTR," the system returned Howard Shore even though I used an abbreviation instead of the full title. This shows that the embedding-based retrieval can recognize some semantic relationships without requiring an exact wording match.

The response to "What is Bilbo Baggins?" (I meant to type Who, but wasn't paying attention) was also correct, even if the answer was very brief. Now, the awards question is very interesting. The system answered 17, which matches the article's number of Academy Awards, but my question only asked about "awards" in general. Since the trilogy received many, many, many other awards, the answer lacked some important context.

The results prove that the system can retrieve and answer straightforward questions fairly well, but there is some wording nuance. Even when an answer is factually related to the source, the model can oversimplify or interpret an ambiguous question more narrowly than intended.

Note: I realized later, after running runnings some the (mostly) successful questions and answers that I should confirm a question that isn't in the material; such as, 'What was Peter Jackson's favororite food while filming Lord of the Rings?' It's not in the wiki and I wanted to test if it said not available or potentially said something like Lembas Bread. It did say Not Available since it wasn't grounded in the material!

## Conclusion

This assignment showed how a basic RAG system can use a Wikipedia article as an external knowledge source instead of a PDF. First the article was converted into text, divided into chunks, represented with embeddings, searched with FAISS, and then used as context for FLAN-T5.

The results showed both the strengths and limitations of the approach. The system answered several questions correctly, including questions that used "LOTR" instead of the full title. It also correctly refused to answer a question when the information wasn't available in the source.

The biggest weakness showed up in the retrieval. The article contained the filming dates, but the correct chunk ranked too low to reach the language model. In another test, weak retrieval led FLAN-T5 to return an incorrect year from unrelated context. This shows that RAG doesn't automatically guarantee an accurate response and the quality of the final answer depends heavily on the chunking strategy, retrieval results, and how well the language model uses the context it receives.

 I'd say the system worked well overall for a simple RAG, but the testing showed some areas that could be improved, especially chunking and retrieval. This is hard work, and I now have a huge appreciation for people who do this at scale.